# Teste isolado — ANATEL (Notícias)

Fonte candidata: **ANATEL - Agência Nacional de Telecomunicações**, setor
Telecom. Notebook **descartável** (Fase 1) — sem dispatcher, sem
`atualizar_status_fonte`, sem gravar nada. Só valida:

1. Listagem das notícias (categoria, título, data, link)
2. Extração do texto completo de uma notícia individual

## Não assumido — confirmado que é DIFERENTE de ANA/ANP

O pedido esperava "mesmo padrão de ANA/ANP" (gov.br/Plone) — **não é**.
O HTML cru da listagem não tem nenhum item (`ul.noticias.listagem-noticias-
com-foto` não existe aqui) porque este é um site **Plone 6 / Volto**
(frontend React, conteúdo em blocos), versão mais nova da mesma família
CMS que ANA/ANP usam, mas arquitetura de fato diferente — mesma situação
que ONS teve com SharePoint vs. o resto do gov.br.

A boa notícia: a API REST pública e padrão do Plone 6 (`++api++/@search`)
funciona sem autenticação, então não precisa raspar HTML nem replicar
proxy nenhum (mais simples que o caso da ONS). Dois endpoints:

| O quê | Endpoint |
|---|---|
| Listagem | `https://www.gov.br/anatel/++api++/@search?portal_type=News+Item&path=/pt-br/assuntos/noticias` |
| Notícia (texto em blocos) | `https://www.gov.br/anatel/++api++/{caminho-da-noticia}` |

`robots.txt` não bloqueia `/pt-br/assuntos/noticias` (só `/search` e
`/login`).

Descoberta importante sobre `path`: tem que ser **relativo à raiz do site**
(`/pt-br/assuntos/noticias`), não `/anatel/pt-br/...` — com o prefixo do
site o filtro silenciosamente devolve 0 resultados, sem erro.

Total: **16 notícias** na pasta — histórico pequeno, sem necessidade de
paginação (`b_size` grande cobre tudo numa chamada só).

In [0]:
%pip install --quiet httpx curl_cffi
dbutils.library.restartPython()

In [0]:
import time
import random
from datetime import datetime
from typing import Optional

import httpx
from curl_cffi import requests as cffi_requests

In [0]:
# =============================================================================
# Configuração
# =============================================================================

BASE_SITE = "https://www.gov.br/anatel/"
BASE_API = "https://www.gov.br/anatel/++api++/"
CAMINHO_NOTICIAS = "/pt-br/assuntos/noticias"

HTTP_TIMEOUT = 30
IMPERSONATE_PROFILES = ["chrome120", "chrome123", "chrome124"]

USER_AGENT = (
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
    "(KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36"
)

HEADERS_API = {
    "User-Agent": USER_AGENT,
    "Accept": "application/json",
    "Accept-Language": "pt-BR,pt;q=0.9",
}


def url_para_api(url_publica: str) -> str:
    """Converte a URL pública de uma notícia na URL da API REST (mesmo
    caminho, prefixado por ++api++ logo após a raiz do site)."""
    if url_publica.startswith(BASE_SITE):
        return url_publica.replace(BASE_SITE, BASE_API, 1)
    return url_publica

In [0]:
def baixar_json(url: str, params: Optional[dict] = None, tentativas: int = 3) -> Optional[dict]:
    for tentativa in range(1, tentativas + 1):
        try:
            resp = httpx.get(url, params=params, headers=HEADERS_API,
                             timeout=HTTP_TIMEOUT, follow_redirects=True)
            if resp.status_code == 200 and resp.text:
                return resp.json()
            print(f"  [httpx tent {tentativa}/{tentativas}] status={resp.status_code}")
        except Exception as e:
            print(f"  [httpx tent {tentativa}/{tentativas}] erro: {e}")

        try:
            impersonate = random.choice(IMPERSONATE_PROFILES)
            resp = cffi_requests.get(url, params=params, headers=HEADERS_API,
                                     impersonate=impersonate, timeout=HTTP_TIMEOUT,
                                     allow_redirects=True)
            if resp.status_code == 200 and resp.text:
                return resp.json()
            print(f"  [curl_cffi tent {tentativa}/{tentativas}] status={resp.status_code}")
        except Exception as e:
            print(f"  [curl_cffi tent {tentativa}/{tentativas}] erro: {e}")

        if tentativa < tentativas:
            time.sleep(random.uniform(1.0, 2.0))

    return None

## Teste 1 — listar as notícias

Filtro `path=/pt-br/assuntos/noticias` (relativo à raiz, sem o prefixo
`/anatel`) + `portal_type=News Item`. `b_size=100` cobre o histórico
inteiro numa chamada só (só 16 itens).

In [0]:
def listar_anatel(max_itens: int = 100) -> list[dict]:
    params = {
        "portal_type": "News Item",
        "path": CAMINHO_NOTICIAS,
        "sort_on": "effective",
        "sort_order": "descending",
        "b_size": max_itens,
    }
    dados = baixar_json(BASE_API + "@search", params=params)
    if not dados:
        return []

    itens = []
    for item in dados.get("items", []):
        data_publicacao = None
        efetiva = item.get("effective")
        if efetiva:
            data_publicacao = efetiva[:10]

        itens.append({
            "titulo": item.get("title"),
            "url": item.get("@id"),
            "published_at": data_publicacao,
        })

    return itens

In [0]:
itens = listar_anatel()

print(f"\n{len(itens)} notícias listadas.\n")
print(f"{'DATA':<12} TÍTULO")
print("-" * 100)
for item in itens:
    print(f"{item['published_at'] or '?':<12} {item['titulo'][:80]}")

urls_unicas = {i["url"] for i in itens}
sem_data = [i for i in itens if not i["published_at"]]

print(f"\nurls únicas: {len(urls_unicas)}/{len(itens)}")
print(f"sem data: {len(sem_data)}")
print(f"\nExemplo de link: {itens[0]['url']}")

## Teste 2 — abrir uma notícia e extrair o texto completo

Conteúdo vem em `blocks` (editor de blocos do Volto/Slate), não texto
corrido — concatena o `plaintext` de cada bloco tipo `slate`, na ordem de
`blocks_layout.items`. `category.nomeCategoria` já vem pronto no JSON.

In [0]:
def extrair_texto_blocks(blocks: dict, ordem: list) -> str:
    partes = []
    for bloco_id in ordem:
        bloco = blocks.get(bloco_id, {})
        if bloco.get("@type") == "slate":
            texto = bloco.get("plaintext", "")
            if texto:
                partes.append(texto)
    return "\n\n".join(partes)


def extrair_noticia_anatel(item: dict) -> Optional[dict]:
    url_api = url_para_api(item["url"])
    dados = baixar_json(url_api)
    if not dados:
        return None

    blocks = dados.get("blocks", {})
    ordem = dados.get("blocks_layout", {}).get("items", [])
    texto = extrair_texto_blocks(blocks, ordem)

    categoria = (dados.get("category") or {}).get("nomeCategoria")
    efetiva = dados.get("effective")

    return {
        "titulo": dados.get("title") or item["titulo"],
        "url": item["url"],
        "categoria": categoria,
        "published_at": (efetiva[:10] if efetiva else item["published_at"]),
        "texto": texto,
    }

In [0]:
AMOSTRA = 6

detalhes = []
for item in itens[:AMOSTRA]:
    print(f"\n  [item] {item['titulo'][:90]}")
    detalhe = extrair_noticia_anatel(item)
    if detalhe is None:
        print("    -> download falhou.")
        continue
    detalhes.append(detalhe)
    print(f"    -> {len(detalhe['texto'])} chars extraídos. categoria={detalhe['categoria']}")
    time.sleep(random.uniform(0.3, 0.8))

print(f"\n{len(detalhes)}/{AMOSTRA} notícias abertas com sucesso.")
curtas = [d for d in detalhes if len(d["texto"]) < 200]
print(f"Com texto abaixo de 200 chars: {len(curtas)}")

In [0]:
# Amostra completa da primeira notícia — pra conferir na mão se bate com o
# que aparece no site.
detalhe = detalhes[0]

print("=" * 100)
print(f"TÍTULO      : {detalhe['titulo']}")
print(f"CATEGORIA   : {detalhe['categoria']}")
print(f"PUBLICADO EM: {detalhe['published_at']}")
print(f"URL         : {detalhe['url']}")
print(f"TAMANHO     : {len(detalhe['texto'])} chars")
print("=" * 100)
print(detalhe["texto"])

## Conclusão da Fase 1

Os dois testes passam: listagem via API devolve todas as 16 notícias com
data, texto completo sai limpo e completo direto dos blocos JSON (sem
precisar de HTML parsing nenhum).

**Avaliação para a Fase 2**: mesmo raciocínio da ONS — é consumo de API
JSON pública via `httpx`, sem Selenium, sem processamento de áudio.
Encaixa no dispatcher genérico `ingest-scraping` com uma `listar_anatel()`
e extrator de blocos próprios (não reaproveita nada de ANA/ANP, que são
HTML/Plone clássico — arquitetura realmente diferente, apesar do nome
"gov.br" em comum).

Histórico pequeno (16 itens) — sem necessidade de paginação nem de limitar
`max_paginas` como em ANP/ABEGÁS.